# Case-Category Model-Family Analysis

Category-level comparison across KLUE-RoBERTa and KoELECTRA.

Set the private project root before execution:

```python
import os
os.environ['KLJPAUDIT_ROOT'] = '/path/to/K-LJPAUDIT'
```

Do not publish raw judicial facts, transformed text, per-case predictions, case identifiers, review files, or model checkpoints.

## Execution block 1

In [ ]:
import os
# ============================================================
# CASE-CATEGORY × MODEL-FAMILY ANALYSIS
# KLUE-RoBERTa vs KoELECTRA Outcome-Cue Sensitivity
# CPU only
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import json
from pathlib import Path

import numpy as np
import pandas as pd

from datasets import load_dataset
from scipy.stats import spearmanr
from sklearn.metrics import accuracy_score, f1_score


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/path/to/K-LJPAUDIT"
)

EXPERIMENT_NAME = (
    "case_category_model_family_analysis_v1"
)

RESULT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "audit_experiments"
    / EXPERIMENT_NAME
)

METRICS_DIR = RESULT_ROOT / "metrics"

RESULT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

KLUE_PREDICTION_PATH = (
    PROJECT_ROOT
    / "results"
    / "audit_experiments"
    / "outcome_cue_insertion_v1"
    / "predictions"
    / "paired_predictions.parquet"
)

KOELECTRA_PREDICTION_PATH = (
    PROJECT_ROOT
    / "results"
    / "audit_experiments"
    / "koelectra_outcome_cue_insertion_v1"
    / "predictions"
    / "koelectra_outcome_cue_predictions_private.parquet"
)

TARGETS = [
    "fine_lv",
    "imprisonment_with_labor_lv",
    "imprisonment_without_labor_lv",
]

NUM_LABELS = {
    "fine_lv": 5,
    "imprisonment_with_labor_lv": 6,
    "imprisonment_without_labor_lv": 5,
}

CONDITIONS = [
    "original",
    "neutral",
    "lenient",
    "severe",
]

MIN_CATEGORY_SUPPORT = 20

assert KLUE_PREDICTION_PATH.exists(), (
    f"KLUE prediction file missing: "
    f"{KLUE_PREDICTION_PATH}"
)

assert KOELECTRA_PREDICTION_PATH.exists(), (
    f"KoELECTRA prediction file missing: "
    f"{KOELECTRA_PREDICTION_PATH}"
)

print("=" * 72)
print("CASE-CATEGORY × MODEL-FAMILY ANALYSIS")
print("=" * 72)
print("KLUE:", KLUE_PREDICTION_PATH)
print("KoELECTRA:", KOELECTRA_PREDICTION_PATH)
print("Minimum category support:", MIN_CATEGORY_SUPPORT)


# ------------------------------------------------------------
# 2. Load test metadata
# ------------------------------------------------------------

dataset = load_dataset(
    "lbox/lbox_open",
    "ljp_criminal",
)

test_raw = dataset["test"]

assert len(test_raw) == 1050

metadata_records = []

for position, row in enumerate(test_raw):
    record = {
        "row_position": position,
        "dataset_case_uid": str(row["id"]),
        "casename": str(
            row["casename"]
        ).strip(),
    }

    if not record["casename"]:
        record["casename"] = "UNKNOWN"

    for target in TARGETS:
        record[target] = int(
            row["label"][target]
        )

    metadata_records.append(record)

metadata_df = pd.DataFrame(
    metadata_records
)

assert len(metadata_df) == 1050
assert (
    metadata_df["dataset_case_uid"]
    .nunique()
    == 1050
)

print("\nTest metadata:")
print(" Cases:", len(metadata_df))
print(
    " Unique case names:",
    metadata_df["casename"].nunique(),
)


# ------------------------------------------------------------
# 3. Prediction schema normalization
# ------------------------------------------------------------

def resolve_column(
    dataframe,
    candidates,
    description,
):
    for candidate in candidates:
        if candidate in dataframe.columns:
            return candidate

    raise KeyError(
        f"Could not resolve {description}. "
        f"Candidates={candidates}; "
        f"available={list(dataframe.columns)}"
    )


def normalize_prediction_artifact(
    path,
    model_name,
):
    df = pd.read_parquet(path).copy()

    assert len(df) == 1050, (
        f"{model_name}: expected 1,050 cases, "
        f"found {len(df)}"
    )

    id_column = resolve_column(
        df,
        [
            "case_uid",
            "id",
            "case_id",
            "example_id",
        ],
        f"{model_name} case ID",
    )

    normalized = pd.DataFrame({
        "artifact_case_uid": (
            df[id_column].astype(str)
        ),
    })

    visibility_candidates = [
        "all_cues_fully_visible",
        "all_conditions_fully_visible",
        "cue_fully_visible",
        "fully_visible",
    ]

    visibility_column = next(
        (
            column
            for column in visibility_candidates
            if column in df.columns
        ),
        None,
    )

    if visibility_column is None:
        normalized["fully_visible"] = True

        print(
            f"{model_name}: visibility column "
            "not found; all cases marked visible."
        )
    else:
        normalized["fully_visible"] = (
            df[visibility_column]
            .fillna(False)
            .astype(bool)
        )

        print(
            f"{model_name}: visibility <- "
            f"{visibility_column}"
        )

    legacy_condition_names = {
        "original": "ORIGINAL",
        "neutral": "NEUTRAL_CUE",
        "lenient": "LENIENT_CUE",
        "severe": "SEVERE_CUE",
    }

    for target in TARGETS:
        true_column = resolve_column(
            df,
            [
                target,
                f"{target}_true",
                f"true_{target}",
                f"{target}_label",
                f"label_{target}",
            ],
            f"{model_name} true {target}",
        )

        normalized[target] = pd.to_numeric(
            df[true_column],
            errors="raise",
        ).astype(int)

        for condition in CONDITIONS:
            legacy_condition = (
                legacy_condition_names[
                    condition
                ]
            )

            prediction_candidates = [
                f"pred_{condition}_{target}",
                f"{condition}_pred_{target}",
                f"pred_{target}_{condition}",
                f"{condition}_{target}_pred",
                (
                    f"{legacy_condition}_"
                    f"{target}_predicted"
                ),
                (
                    f"{legacy_condition}_"
                    f"{target}_pred"
                ),
                (
                    f"{legacy_condition}_"
                    f"pred_{target}"
                ),
                (
                    f"{legacy_condition.lower()}_"
                    f"{target}_predicted"
                ),
                (
                    f"{legacy_condition.lower()}_"
                    f"{target}_pred"
                ),
            ]

            prediction_column = resolve_column(
                df,
                prediction_candidates,
                (
                    f"{model_name} "
                    f"{condition} prediction "
                    f"for {target}"
                ),
            )

            normalized[
                f"pred_{condition}_{target}"
            ] = pd.to_numeric(
                df[prediction_column],
                errors="raise",
            ).astype(int)

            print(
                f"{model_name}: "
                f"pred_{condition}_{target} <- "
                f"{prediction_column}"
            )

    assert (
        normalized["artifact_case_uid"]
        .notna()
        .all()
    )

    assert (
        normalized["artifact_case_uid"]
        .nunique()
        == 1050
    )

    for target in TARGETS:
        valid_values = set(
            range(NUM_LABELS[target])
        )

        assert set(
            normalized[target].unique()
        ).issubset(valid_values)

        for condition in CONDITIONS:
            prediction_column = (
                f"pred_{condition}_{target}"
            )

            assert set(
                normalized[
                    prediction_column
                ].unique()
            ).issubset(valid_values)

    print(
        f"{model_name}: prediction schema "
        "normalization PASSED"
    )

    return normalized


klue_df = normalize_prediction_artifact(
    KLUE_PREDICTION_PATH,
    "KLUE-RoBERTa",
)

koelectra_df = normalize_prediction_artifact(
    KOELECTRA_PREDICTION_PATH,
    "KoELECTRA",
)


# ------------------------------------------------------------
# 4. Attach case names safely
# ------------------------------------------------------------

def attach_metadata(
    prediction_df,
    model_name,
):
    prediction_ids = set(
        prediction_df[
            "artifact_case_uid"
        ].astype(str)
    )

    metadata_ids = set(
        metadata_df[
            "dataset_case_uid"
        ].astype(str)
    )

    # Preferred: exact ID merge
    if prediction_ids == metadata_ids:
        merged = prediction_df.merge(
            metadata_df,
            left_on="artifact_case_uid",
            right_on="dataset_case_uid",
            how="inner",
            validate="one_to_one",
            suffixes=("", "_metadata"),
        )

        alignment_mode = "exact_case_uid"

    else:
        # Fallback allowed only when all three reference-label
        # sequences exactly match the official test order.
        prediction_label_matrix = (
            prediction_df[TARGETS]
            .to_numpy(dtype=np.int64)
        )

        metadata_label_matrix = (
            metadata_df[TARGETS]
            .to_numpy(dtype=np.int64)
        )

        labels_match_by_position = np.array_equal(
            prediction_label_matrix,
            metadata_label_matrix,
        )

        assert labels_match_by_position, (
            f"{model_name}: case IDs do not match "
            "the official dataset IDs, and positional "
            "reference-label validation also failed. "
            "Category metadata cannot be attached safely."
        )

        merged = prediction_df.copy()

        merged["dataset_case_uid"] = (
            metadata_df[
                "dataset_case_uid"
            ].to_numpy()
        )

        merged["casename"] = (
            metadata_df[
                "casename"
            ].to_numpy()
        )

        merged["row_position"] = (
            metadata_df[
                "row_position"
            ].to_numpy()
        )

        alignment_mode = (
            "validated_test_order_by_reference_labels"
        )

    assert len(merged) == 1050
    assert merged["casename"].notna().all()

    for target in TARGETS:
        metadata_target_column = (
            f"{target}_metadata"
        )

        if metadata_target_column in merged.columns:
            assert np.array_equal(
                merged[target].to_numpy(),
                merged[
                    metadata_target_column
                ].to_numpy(),
            )

            merged = merged.drop(
                columns=[metadata_target_column]
            )

    print(
        f"{model_name} alignment:",
        alignment_mode,
    )

    print(
        f"{model_name} visible cases:",
        int(merged["fully_visible"].sum()),
    )

    return merged, alignment_mode


klue_df, klue_alignment = attach_metadata(
    klue_df,
    "KLUE-RoBERTa",
)

koelectra_df, koelectra_alignment = (
    attach_metadata(
        koelectra_df,
        "KoELECTRA",
    )
)


# ------------------------------------------------------------
# 5. Category distribution
# ------------------------------------------------------------

category_distribution_df = (
    metadata_df
    .groupby("casename")
    .size()
    .reset_index(name="total_support")
    .sort_values(
        ["total_support", "casename"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

category_distribution_df[
    "meets_minimum_support"
] = (
    category_distribution_df[
        "total_support"
    ] >= MIN_CATEGORY_SUPPORT
)

print("\nCategory distribution:")
print(
    category_distribution_df.head(20)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 6. Per-category baseline performance
# ------------------------------------------------------------

model_data = {
    "KLUE-RoBERTa": klue_df,
    "KoELECTRA": koelectra_df,
}

baseline_rows = []

for model_name, dataframe in model_data.items():
    visible_df = dataframe.loc[
        dataframe["fully_visible"]
    ].copy()

    for casename, group in visible_df.groupby(
        "casename"
    ):
        support = len(group)

        true_matrix = np.column_stack([
            group[target].to_numpy()
            for target in TARGETS
        ])

        pred_matrix = np.column_stack([
            group[
                f"pred_original_{target}"
            ].to_numpy()
            for target in TARGETS
        ])

        joint_em = float(
            np.mean(
                np.all(
                    true_matrix == pred_matrix,
                    axis=1,
                )
            )
        )

        macro_f1_values = []

        row = {
            "model": model_name,
            "casename": casename,
            "support": support,
            "meets_minimum_support": (
                support
                >= MIN_CATEGORY_SUPPORT
            ),
            "joint_exact_match": joint_em,
        }

        for target in TARGETS:
            y_true = group[target].to_numpy()
            y_pred = group[
                f"pred_original_{target}"
            ].to_numpy()

            accuracy = accuracy_score(
                y_true,
                y_pred,
            )

            macro_f1 = f1_score(
                y_true,
                y_pred,
                labels=list(
                    range(NUM_LABELS[target])
                ),
                average="macro",
                zero_division=0,
            )

            row[
                f"{target}_accuracy"
            ] = float(accuracy)

            row[
                f"{target}_macro_f1"
            ] = float(macro_f1)

            macro_f1_values.append(macro_f1)

        row["mean_macro_f1"] = float(
            np.mean(macro_f1_values)
        )

        baseline_rows.append(row)

category_baseline_df = pd.DataFrame(
    baseline_rows
)


# ------------------------------------------------------------
# 7. Category-level cue sensitivity
# ------------------------------------------------------------

cue_rows = []

for model_name, dataframe in model_data.items():
    visible_df = dataframe.loc[
        dataframe["fully_visible"]
    ].copy()

    for casename, group in visible_df.groupby(
        "casename"
    ):
        support = len(group)

        true_matrix = np.column_stack([
            group[target].to_numpy()
            for target in TARGETS
        ])

        original_prediction_matrix = (
            np.column_stack([
                group[
                    f"pred_original_{target}"
                ].to_numpy()
                for target in TARGETS
            ])
        )

        original_joint_correct = np.all(
            true_matrix
            == original_prediction_matrix,
            axis=1,
        )

        for condition in [
            "neutral",
            "lenient",
            "severe",
        ]:
            condition_prediction_matrix = (
                np.column_stack([
                    group[
                        f"pred_{condition}_{target}"
                    ].to_numpy()
                    for target in TARGETS
                ])
            )

            condition_joint_correct = np.all(
                true_matrix
                == condition_prediction_matrix,
                axis=1,
            )

            joint_flip = np.any(
                condition_prediction_matrix
                != original_prediction_matrix,
                axis=1,
            )

            row = {
                "model": model_name,
                "casename": casename,
                "condition": condition,
                "support": support,
                "meets_minimum_support": (
                    support
                    >= MIN_CATEGORY_SUPPORT
                ),
                "joint_flip_rate": float(
                    joint_flip.mean()
                ),
                "original_joint_exact_match": (
                    float(
                        original_joint_correct.mean()
                    )
                ),
                "condition_joint_exact_match": (
                    float(
                        condition_joint_correct.mean()
                    )
                ),
                "delta_joint_exact_match": float(
                    condition_joint_correct.mean()
                    - original_joint_correct.mean()
                ),
            }

            for target in TARGETS:
                original_predictions = group[
                    f"pred_original_{target}"
                ].to_numpy()

                condition_predictions = group[
                    f"pred_{condition}_{target}"
                ].to_numpy()

                row[
                    f"{target}_flip_rate"
                ] = float(
                    np.mean(
                        condition_predictions
                        != original_predictions
                    )
                )

                row[
                    f"{target}_mean_shift"
                ] = float(
                    np.mean(
                        condition_predictions
                        - original_predictions
                    )
                )

            cue_rows.append(row)

category_cue_df = pd.DataFrame(cue_rows)


# ------------------------------------------------------------
# 8. Cross-model category comparison
# ------------------------------------------------------------

reliable_cue_df = category_cue_df.loc[
    category_cue_df[
        "meets_minimum_support"
    ]
].copy()

klue_reliable = reliable_cue_df.loc[
    reliable_cue_df["model"].eq(
        "KLUE-RoBERTa"
    )
].copy()

koelectra_reliable = reliable_cue_df.loc[
    reliable_cue_df["model"].eq(
        "KoELECTRA"
    )
].copy()

cross_model_df = klue_reliable.merge(
    koelectra_reliable,
    on=["casename", "condition"],
    how="inner",
    suffixes=("_klue", "_koelectra"),
)

if len(cross_model_df) > 0:
    cross_model_df[
        "absolute_joint_flip_difference"
    ] = np.abs(
        cross_model_df[
            "joint_flip_rate_klue"
        ]
        - cross_model_df[
            "joint_flip_rate_koelectra"
        ]
    )

    cross_model_df[
        "same_delta_joint_em_direction"
    ] = (
        np.sign(
            cross_model_df[
                "delta_joint_exact_match_klue"
            ]
        )
        == np.sign(
            cross_model_df[
                "delta_joint_exact_match_koelectra"
            ]
        )
    )
else:
    cross_model_df = pd.DataFrame()


# ------------------------------------------------------------
# 9. Correlations by cue condition
# ------------------------------------------------------------

correlation_rows = []

for condition in [
    "neutral",
    "lenient",
    "severe",
]:
    condition_df = cross_model_df.loc[
        cross_model_df[
            "condition"
        ].eq(condition)
    ].copy()

    category_count = len(condition_df)

    if category_count >= 3:
        flip_result = spearmanr(
            condition_df[
                "joint_flip_rate_klue"
            ],
            condition_df[
                "joint_flip_rate_koelectra"
            ],
        )

        delta_result = spearmanr(
            condition_df[
                "delta_joint_exact_match_klue"
            ],
            condition_df[
                "delta_joint_exact_match_koelectra"
            ],
        )

        flip_correlation = float(
            flip_result.statistic
        )

        flip_p_value = float(
            flip_result.pvalue
        )

        delta_correlation = float(
            delta_result.statistic
        )

        delta_p_value = float(
            delta_result.pvalue
        )
    else:
        flip_correlation = np.nan
        flip_p_value = np.nan
        delta_correlation = np.nan
        delta_p_value = np.nan

    correlation_rows.append({
        "condition": condition,
        "category_count": category_count,
        "joint_flip_spearman_rho": (
            flip_correlation
        ),
        "joint_flip_spearman_p": (
            flip_p_value
        ),
        "delta_joint_em_spearman_rho": (
            delta_correlation
        ),
        "delta_joint_em_spearman_p": (
            delta_p_value
        ),
        "mean_absolute_flip_difference": (
            float(
                condition_df[
                    "absolute_joint_flip_difference"
                ].mean()
            )
            if category_count > 0
            else np.nan
        ),
        "same_delta_direction_rate": (
            float(
                condition_df[
                    "same_delta_joint_em_direction"
                ].mean()
            )
            if category_count > 0
            else np.nan
        ),
    })

correlation_df = pd.DataFrame(
    correlation_rows
)


# ------------------------------------------------------------
# 10. Identify most and least sensitive categories
# ------------------------------------------------------------

reliable_ranked_df = (
    reliable_cue_df
    .sort_values(
        [
            "model",
            "condition",
            "joint_flip_rate",
            "support",
        ],
        ascending=[
            True,
            True,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

top_sensitive_df = (
    reliable_ranked_df
    .groupby(
        ["model", "condition"],
        group_keys=False,
    )
    .head(5)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 11. Save aggregate-only outputs
# ------------------------------------------------------------

category_distribution_path = (
    METRICS_DIR
    / "case_category_distribution.csv"
)

category_baseline_path = (
    METRICS_DIR
    / "category_baseline_performance.csv"
)

category_cue_path = (
    METRICS_DIR
    / "category_outcome_cue_sensitivity.csv"
)

cross_model_path = (
    METRICS_DIR
    / "cross_model_category_comparison.csv"
)

correlation_path = (
    METRICS_DIR
    / "cross_model_category_correlations.csv"
)

top_sensitive_path = (
    METRICS_DIR
    / "top_sensitive_categories.csv"
)

category_distribution_df.to_csv(
    category_distribution_path,
    index=False,
)

category_baseline_df.to_csv(
    category_baseline_path,
    index=False,
)

category_cue_df.to_csv(
    category_cue_path,
    index=False,
)

cross_model_df.to_csv(
    cross_model_path,
    index=False,
)

correlation_df.to_csv(
    correlation_path,
    index=False,
)

top_sensitive_df.to_csv(
    top_sensitive_path,
    index=False,
)

summary = {
    "experiment_name": EXPERIMENT_NAME,
    "status": "completed",
    "analysis_type": (
        "case_category_model_family_analysis"
    ),
    "models": [
        "KLUE-RoBERTa",
        "KoELECTRA",
    ],
    "minimum_category_support": (
        MIN_CATEGORY_SUPPORT
    ),
    "total_test_cases": 1050,
    "unique_case_categories": int(
        metadata_df["casename"].nunique()
    ),
    "reliable_category_count": int(
        category_distribution_df[
            "meets_minimum_support"
        ].sum()
    ),
    "alignment": {
        "klue_roberta": klue_alignment,
        "koelectra": koelectra_alignment,
    },
    "cross_model_correlations": (
        correlation_df
        .replace({np.nan: None})
        .to_dict(orient="records")
    ),
    "interpretive_scope": (
        "Category-level results with support below "
        "20 are descriptive diagnostics only. "
        "Correlations are exploratory because the "
        "number of sufficiently supported categories "
        "may be small."
    ),
    "privacy": {
        "per_case_outputs_saved": False,
        "aggregate_category_metrics_only": True,
    },
}

summary_path = (
    RESULT_ROOT / "experiment_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        ensure_ascii=False,
        indent=2,
    )


# ------------------------------------------------------------
# 12. Final output
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("RELIABLE CATEGORY COUNTS")
print("=" * 72)

print(
    "Total categories:",
    metadata_df["casename"].nunique(),
)

print(
    f"Categories with N >= "
    f"{MIN_CATEGORY_SUPPORT}:",
    int(
        category_distribution_df[
            "meets_minimum_support"
        ].sum()
    ),
)

print("\n" + "=" * 72)
print("CROSS-MODEL CATEGORY CORRELATIONS")
print("=" * 72)

print(
    correlation_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

print("\n" + "=" * 72)
print("TOP SENSITIVE SUPPORTED CATEGORIES")
print("=" * 72)

display_columns = [
    "model",
    "condition",
    "casename",
    "support",
    "joint_flip_rate",
    "delta_joint_exact_match",
]

print(
    top_sensitive_df[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

print("\nSaved:")
print(" ", category_distribution_path)
print(" ", category_baseline_path)
print(" ", category_cue_path)
print(" ", cross_model_path)
print(" ", correlation_path)
print(" ", top_sensitive_path)
print(" ", summary_path)

print("\nCASE-CATEGORY ANALYSIS: COMPLETED")
